# Error taxonomy

Goal: find which kinds of financial language the frozen FinBERT CNN gets wrong and group them into an error taxonomy

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

sys.path.append("..")

pd.set_option("display.max_colwidth", None)
os.makedirs("../results/taxonomy", exist_ok=True)

## 1. Error pool
References from notebook2

In [2]:
train = pd.read_csv("../data/train.csv")
cnn = pd.read_csv("../results/hard_samples/band_train.csv")
assert (cnn.sentence.values == train.sentence.values).all()

errors = cnn[["sentence", "label", "pred", "max_prob", "is_hard", "all_agree"]].copy()
errors.insert(0, "train_idx", np.arange(len(train)))
errors = errors.rename(columns={"pred": "cnn_pred", "max_prob": "cnn_max_prob"})
errors["cnn_max_prob"] = errors.cnn_max_prob.round(3)
errors["n_words"] = train.sentence.str.split().str.len()

pool = errors[errors.cnn_pred != errors.label].copy()
pool["error_type"] = pool.label + " -> " + pool.cnn_pred
pool.to_csv("../results/taxonomy/error_pool.csv", index=False)

print(f"CNN wrong (pool): {len(pool)} / {len(train)}, of which hard: {pool.is_hard.sum()}, all-agree: {pool.all_agree.sum()}")
pool.error_type.value_counts()

CNN wrong (pool): 615 / 2901, of which hard: 193, all-agree: 114


error_type
positive -> neutral     273
neutral -> positive     123
negative -> neutral     100
negative -> positive     52
neutral -> negative      45
positive -> negative     22
Name: count, dtype: int64

## 2. Scheme coded

In [ ]:
CATEGORIES = ["ambiguous_label", "domain_required", "mixed_sentiment", "corporate_action",
              "numerical_comparison", "other"]
CODERS = ["A", "B", "C", "D"]
N_UNIQUE, N_SHARED = 50, 20

sample, _ = train_test_split(pool, train_size=N_SHARED + N_UNIQUE * len(CODERS),
                             stratify=pool.error_type, random_state=42)
sample = sample.sample(frac=1, random_state=42)
shared = sample.iloc[:N_SHARED]
unique = sample.iloc[N_SHARED:]

columns = ["train_idx", "sentence", "label", "cnn_pred"]
for i, coder in enumerate(CODERS):
    path = f"../results/taxonomy/coding_sheet_{coder}.csv"
    if os.path.exists(path):
        continue
    sheet = pd.concat([shared, unique.iloc[i * N_UNIQUE:(i + 1) * N_UNIQUE]])[columns]
    sheet = sheet.sample(frac=1, random_state=i)
    sheet["shared"] = sheet.train_idx.isin(shared.train_idx)
    for category in CATEGORIES:
        sheet[category] = ""
    sheet["notes"] = ""
    sheet.to_csv(path, index=False)

print(f"{len(CODERS)} sheets x {N_UNIQUE + N_SHARED} sentences ({N_SHARED} shared), "
      f"{len(sample)} distinct sentences out of {len(pool)} in the pool")
sample.error_type.value_counts()

4 sheets x 70 sentences (20 shared), 220 distinct sentences out of 615 in the pool


error_type
positive -> neutral     98
neutral -> positive     44
negative -> neutral     36
negative -> positive    18
neutral -> negative     16
positive -> negative     8
Name: count, dtype: int64

## Eval

In [ ]:
sheets = {c: pd.read_csv(f"../results/taxonomy/coding_sheet_{c}.csv") for c in CODERS}
shared_coded = {c: s[s.shared].set_index("train_idx").sort_index()[CATEGORIES].fillna(0).astype(int)
                for c, s in sheets.items()}

if all(s.to_numpy().sum() > 0 for s in shared_coded.values()):
    pairs = [(a, b) for i, a in enumerate(CODERS) for b in CODERS[i + 1:]]
    kappa = pd.Series({cat: np.mean([cohen_kappa_score(shared_coded[a][cat], shared_coded[b][cat]) for a, b in pairs])
                       for cat in CATEGORIES})
    display(kappa.round(2).sort_values())